## 01: Data preprocessing
Reads the raw 2024 BRFSS file, cleans it, builds the feature lists and subgroup columns, makes the 70/10/20 split (seed 75), runs leakage checks 1 and 2, and saves the split files to `data/derived/`.

**Before running:** unzip `data/raw/LLCP2024XPT.zip` in `data/raw/`. The unzipped file is `LLCP2024.XPT ` **with a trailing space**; keep that name. The split files are already in `data/derived/`, so rerunning this notebook is only needed to check the rebuild (it overwrites them with identical content). Comments that cite `INSTRUCTIONS.md` refer to our internal planning notes (not in this repository); the choices are explained in our report.

(Originally the course's Week 6 preprocessing notebook.)

In [1]:
# Step 1: Load the raw data
# Runs locally (no Colab). The raw zip was downloaded from cdc.gov into data/raw/ and unzipped there.
# Note: the file name inside the CDC zip really ends in a space ("LLCP2024.XPT "); we keep it as-is
# because files in data/raw/ are never edited.
import os
import pandas as pd

RAW = os.path.join("data", "raw", "LLCP2024.XPT ")
DERIVED = os.path.join("data", "derived")
OUTPUTS = "outputs"
os.makedirs(DERIVED, exist_ok=True)
os.makedirs(OUTPUTS, exist_ok=True)

df = pd.read_sas(RAW, format="xport", encoding="latin-1")
print(df.shape)   # expect (457670, 301)

(457670, 301)


In [2]:
# Choose columns to keep
keep = ["SEQNO","_STATE","DIABETE4",
        "_AGEG5YR","SEXVAR","_RACEGR3","_EDUCAG","_INCOMG1","_URBSTAT",
        "_BMI5","_SMOKER3","EXERANY2","ALCDAY4",
        "_HLTHPL2","PERSDOC3","MEDCOST1","CHECKUP1",
        "GENHLTH","PHYSHLTH","MENTHLTH",
        "SDLONELY","SDHEMPLY","FOODSTMP","SDHFOOD1","SDHBILLS","SDHUTILS","SDHTRNSP"]
slim = df[keep].copy()

# Save the slim copy locally (replaces the old Google Drive save)
slim.to_csv(os.path.join(DERIVED, "brfss2024_slim.csv"), index=False)
print(slim.shape)

(457670, 27)


In [3]:
# Pre-processing - print variable's values
# Start with initial features
features = ["_AGEG5YR","SEXVAR","_RACEGR3","_EDUCAG","_INCOMG1","_URBSTAT",
            "_BMI5","_SMOKER3","EXERANY2","ALCDAY4",
            "_HLTHPL2","PERSDOC3","MEDCOST1","CHECKUP1",
            "GENHLTH","PHYSHLTH","MENTHLTH"]

for c in features:
  n = slim[c].nunique() # number of differet values per variable
  if n <= 40:
    print(c, slim[c].value_counts(dropna=False).sort_index().to_dict())
  else:
    print(c, " | min:", slim[c].min(), " | max:", slim[c].max(), "| blank share:", round(slim[c].isna().mean(), 3))

_AGEG5YR {1.0: 29692, 2.0: 23705, 3.0: 26237, 4.0: 28804, 5.0: 30899, 6.0: 28968, 7.0: 31698, 8.0: 34936, 9.0: 43387, 10.0: 47701, 11.0: 44774, 12.0: 36803, 13.0: 41756, 14.0: 8310}
SEXVAR {1.0: 217487, 2.0: 240183}
_RACEGR3 {1.0: 329346, 2.0: 35172, 3.0: 24917, 4.0: 10486, 5.0: 48646, 9.0: 9103}
_EDUCAG {1.0: 27149, 2.0: 115877, 3.0: 120732, 4.0: 191549, 9.0: 2363}
_INCOMG1 {1.0: 20800, 2.0: 33503, 3.0: 41702, 4.0: 49686, 5.0: 112409, 6.0: 82453, 7.0: 29694, 9.0: 87423}
_URBSTAT {1.0: 383253, 2.0: 59794, nan: 14623}
_BMI5  | min: 1200.0  | max: 9984.0 | blank share: 0.094
_SMOKER3 {1.0: 33118, 2.0: 13976, 3.0: 119598, 4.0: 258956, 9.0: 32022}
EXERANY2 {1.0: 350061, 2.0: 106294, 7.0: 860, 9.0: 452, nan: 3}
ALCDAY4  | min: 100.0  | max: 999.0 | blank share: 0.086
_HLTHPL2 {1.0: 413720, 2.0: 25406, 9.0: 18544}
PERSDOC3 {1.0: 242393, 2.0: 154558, 3.0: 56106, 7.0: 3446, 9.0: 1164, nan: 3}
MEDCOST1 {1.0: 43363, 2.0: 412634, 7.0: 1229, 9.0: 438, nan: 6}
CHECKUP1 {1.0: 370023, 2.0: 39655, 3.0

In [4]:
import numpy as np

# keeping only 1 (yes), 3 (no)
d = slim[slim["DIABETE4"].isin([1,3])].copy()
d["target"] = (d["DIABETE4"] == 1).astype(int)

print("Labeled rows:", len(d))

# drop 14 (unknown)
d = d[d["_AGEG5YR"] != 14].copy()
print("After dropping unknown age:", len(d), "| positive rate:", d["target"].mean().round(3))

# 7 = don't know, 9 = refused
for c in ["EXERANY2", "MEDCOST1", "PERSDOC3", "GENHLTH", "CHECKUP1"]:
    d[c] = d[c].replace({7: np.nan, 9: np.nan})

# for these calculated variables, 9 = "don't know / refused / missing"
for c in ["_RACEGR3", "_EDUCAG", "_INCOMG1", "_SMOKER3", "_HLTHPL2"]:
    d[c] = d[c].replace({9: np.nan})

# days not good: 77 and 99 become missing, 88 ("none") becomes 0 days
for c in ["PHYSHLTH", "MENTHLTH"]:
    d[c] = d[c].replace({77: np.nan, 99: np.nan, 88: 0})

# BMI
d["BMI"] = d["_BMI5"] / 100
# clip values to a fixed range of 15-60
# - anything > BMI of 60 becomes 60
# - anything < BMI of 15 becomes 15
d["BMI"] = d["BMI"].clip(lower=15, upper=60)
print(d["BMI"].describe().round(1))

# alcohol - convert everything to per-30 day answers
a = d["ALCDAY4"]
d["ALC_DAYS30"] = np.select(
    [a.between(101, 107), a.between(201, 230), a == 888],
    [(a - 100) * 30 / 7,   a - 200,            0],
    default=np.nan,
)
print(d[["BMI", "ALC_DAYS30"]].describe().round(1))

Labeled rows: 441934
After dropping unknown age: 433963 | positive rate: 0.149


count    396239.0
mean         28.5
std           6.4
min          15.0
25%          24.1
50%          27.4
75%          31.6
max          60.0
Name: BMI, dtype: float64
            BMI  ALC_DAYS30
count  396239.0    393706.0
mean       28.5         4.7
std         6.4         8.0
min        15.0         0.0
25%        24.1         0.0
50%        27.4         1.0
75%        31.6         5.0
max        60.0        30.0


In [5]:
# BMI check
print(d["BMI"].quantile([0.001, 0.01, 0.5, 0.99, 0.999]).round(1))
print("BMI above 60:", (d["BMI"] > 60).sum(), "| below 15:", (d["BMI"] < 15).sum())

0.001    15.0
0.010    17.7
0.500    27.4
0.990    49.4
0.999    60.0
Name: BMI, dtype: float64
BMI above 60: 0 | below 15: 0


In [6]:
# Pre-processing of SDOH items
sd_scale = ["SDLONELY", "SDHFOOD1"]
sd_yesno = ["SDHEMPLY", "FOODSTMP", "SDHBILLS", "SDHUTILS", "SDHTRNSP"]
sd_all = sd_scale + sd_yesno

# Marks respondents who were asked the SDOH module
d["has_sdoh"] = d[sd_all].notna().any(axis=1)

# Replace 7 (don't know) and 9 (refused) with blank
for c in sd_scale:
    d[c] = d[c].replace({7: np.nan, 9: np.nan})

# Replace 7 (don't know) and 9 (refused) with blank
# Also change 2 (no) to 0 -> 1 yes, 0 no
for c in sd_yesno:
    d[c] = d[c].replace({7: np.nan, 9: np.nan, 2: 0})   # 1 = yes, 0 = no

print("Module respondents:", d["has_sdoh"].sum())

Module respondents: 192706


In [7]:
# Check and save
feature_cols = ["_AGEG5YR","SEXVAR","_RACEGR3","_EDUCAG","_INCOMG1","_URBSTAT",
                "BMI","_SMOKER3","EXERANY2","ALC_DAYS30",
                "_HLTHPL2","PERSDOC3","MEDCOST1","CHECKUP1",
                "GENHLTH","PHYSHLTH","MENTHLTH"]
print((d[feature_cols].isna().mean() * 100).round(1).sort_values(ascending=False))
d.to_csv(os.path.join(DERIVED, "brfss2024_clean.csv"), index=False)

# Save clean file
clean = d.copy()

_INCOMG1      18.3
ALC_DAYS30     9.3
BMI            8.7
_SMOKER3       6.7
_HLTHPL2       3.8
_URBSTAT       3.2
PHYSHLTH       2.3
MENTHLTH       1.7
_RACEGR3       1.7
CHECKUP1       1.1
PERSDOC3       0.9
_EDUCAG        0.4
GENHLTH        0.3
EXERANY2       0.3
MEDCOST1       0.3
SEXVAR         0.0
_AGEG5YR       0.0
dtype: float64


In [8]:
# Define feature lists -> different models (M0, M1, M2, M3)
age_sex = ["_AGEG5YR", "SEXVAR"]
socio   = ["_RACEGR3", "_EDUCAG", "_INCOMG1", "_URBSTAT"]
other   = ["BMI", "_SMOKER3", "EXERANY2", "ALC_DAYS30", "_HLTHPL2", "PERSDOC3",
           "MEDCOST1", "CHECKUP1", "GENHLTH", "PHYSHLTH", "MENTHLTH"]
sdoh    = ["SDLONELY", "SDHEMPLY", "FOODSTMP", "SDHFOOD1", "SDHBILLS", "SDHUTILS", "SDHTRNSP"]

M0 = age_sex
M1 = M0 + socio
M2 = M1 + other
M3 = M2 + sdoh
print(len(M0), len(M1), len(M2), len(M3))

2 6 17 24


In [9]:
# Create subgroup columns (for testing models on different subgroups)
clean["grp_age"]    = pd.cut(clean["_AGEG5YR"], bins=[0, 3, 6, 9, 13],
                             labels=["18-34", "35-49", "50-64", "65+"])
clean["grp_sex"]    = clean["SEXVAR"].map({1: "Male", 2: "Female"})
clean["grp_race"]   = clean["_RACEGR3"].map({1: "White", 2: "Black", 3: "Other race",
                                             4: "Multiracial", 5: "Hispanic"}).fillna("Unknown")
clean["grp_income"] = pd.cut(clean["_INCOMG1"], bins=[0, 3, 5, 7],
                             labels=["Under $35k", "$35k-$100k", "$100k+"]).astype(object).fillna("Unknown")
clean["grp_urban"]  = clean["_URBSTAT"].map({1: "Urban", 2: "Rural"}).fillna("Unknown")

for g in ["grp_age", "grp_sex", "grp_race", "grp_income", "grp_urban"]:
    print(clean.groupby(g, observed=True)["target"].agg(["count", "sum", "mean"]).round(3), "\n")

          count    sum   mean
grp_age                      
18-34     77864   1616  0.021
35-49     85473   6471  0.076
50-64    105619  19354  0.183
65+      165007  37380  0.227 

          count    sum   mean
grp_sex                      
Female   225752  32420  0.144
Male     208211  32401  0.156 

              count    sum   mean
grp_race                         
Black         32795   7005  0.214
Hispanic      45571   6427  0.141
Multiracial    9858   1375  0.139
Other race    23200   3619  0.156
Unknown        7204   1204  0.167
White        315335  45191  0.143 

             count    sum   mean
grp_income                      
$100k+      108523  10022  0.092
$35k-$100k  155423  23444  0.151
Under $35k   90750  19592  0.216
Unknown      79267  11763  0.148 

            count    sum   mean
grp_urban                      
Rural       57008   9772  0.171
Unknown     13709   2288  0.167
Urban      363246  52761  0.145 



## Split (step 3)
Fresh 70/10/20 split, stratified on the diabetes label, seed 75 (decision R3). Seed 74 from the old build is superseded:
reusing it would reproduce the old, already-seen test rows.
Test rows go to a separate, clearly labeled file that no code opens until the single final evaluation (decision R9).

In [10]:
# Split data: 70% train, 10% validation, 20% test, stratified on the label
from sklearn.model_selection import train_test_split

SEED = 75   # approved split seed (R3); has no special meaning, it only makes the shuffle repeatable

train_val, test = train_test_split(clean, test_size=0.20, stratify=clean["target"], random_state=SEED)
train, val = train_test_split(train_val, test_size=0.125, stratify=train_val["target"], random_state=SEED)
# 0.125 of the remaining 80% = 10% of all rows

clean["split"] = "train"
clean.loc[val.index, "split"] = "val"
clean.loc[test.index, "split"] = "test"
print(clean.shape)   # expect (433963, 37)

(433963, 37)


In [11]:
# Leakage checks 1 and 2 (INSTRUCTIONS.md section 6). These need all three parts, so they run here.
ids = {s: set(zip(clean.loc[clean["split"] == s, "_STATE"], clean.loc[clean["split"] == s, "SEQNO"]))
       for s in ["train", "val", "test"]}
n_dup = int(clean[["_STATE", "SEQNO"]].duplicated().sum())
c1 = {
    "no duplicate (_STATE, SEQNO) pairs":     n_dup == 0,
    "train and val disjoint":                 ids["train"].isdisjoint(ids["val"]),
    "train and test disjoint":                ids["train"].isdisjoint(ids["test"]),
    "val and test disjoint":                  ids["val"].isdisjoint(ids["test"]),
    "splits together cover every kept row":   sum(len(v) for v in ids.values()) == len(clean),
}
print("Check 1: identifiers")
for k, v in c1.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")

print("\nCheck 2: sizes and positive rate by split")
c2 = clean.groupby("split")["target"].agg(n="count", positives="sum", positive_rate="mean")
c2["share_of_rows"] = c2["n"] / len(clean)
c2 = c2.loc[["train", "val", "test"]]
print(c2.round(4))
c2_ok = (c2["share_of_rows"] - pd.Series({"train": .7, "val": .1, "test": .2})).abs().max() < 0.001 \
        and (c2["positive_rate"].max() - c2["positive_rate"].min()) < 0.001
print("  PASS" if c2_ok else "  FAIL", " sizes about 70/10/20 and positive rate about equal")

print("\nSDOH module respondents by split")
sd = clean.groupby("split")["has_sdoh"].agg(n="count", module="sum", share="mean").loc[["train", "val", "test"]]
sd["positive_rate_in_module"] = clean[clean["has_sdoh"]].groupby("split")["target"].mean()
print(sd.round(4))

pd.DataFrame([{"check": k, "pass": v} for k, v in c1.items()]
             + [{"check": "sizes ~70/10/20 and equal positive rate", "pass": bool(c2_ok)}]
            ).to_csv(os.path.join(OUTPUTS, "step3_leakage_checks_1_2.csv"), index=False)
c2.join(sd[["module", "share", "positive_rate_in_module"]]).to_csv(
    os.path.join(OUTPUTS, "step3_split_sizes.csv"))
assert all(c1.values()) and c2_ok, "LEAKAGE CHECK FAILED: stop and report"

Check 1: identifiers
  PASS  no duplicate (_STATE, SEQNO) pairs
  PASS  train and val disjoint
  PASS  train and test disjoint
  PASS  val and test disjoint
  PASS  splits together cover every kept row

Check 2: sizes and positive rate by split
            n  positives  positive_rate  share_of_rows
split                                                 
train  303773      45375         0.1494            0.7
val     43397       6482         0.1494            0.1
test    86793      12964         0.1494            0.2
  PASS  sizes about 70/10/20 and positive rate about equal

SDOH module respondents by split
            n  module   share  positive_rate_in_module
split                                                 
train  303773  134963  0.4443                   0.1547
val     43397   19387  0.4467                   0.1555
test    86793   38356  0.4419                   0.1543


In [12]:
# Save the split (R9): test rows in their own clearly labeled file; train + validation in the dev file.
# 02_Model_Training.ipynb reads ONLY the dev file. The test file is opened once, at the final evaluation (step 7).
TEST_FILE = os.path.join(DERIVED, "TEST_SET_DO_NOT_OPEN_until_final_evaluation.csv")
DEV_FILE  = os.path.join(DERIVED, "dev_set_train_and_validation.csv")

clean[clean["split"] == "test"].to_csv(TEST_FILE, index=False)
clean[clean["split"] != "test"].to_csv(DEV_FILE, index=False)
clean[["_STATE", "SEQNO", "split"]].to_csv(os.path.join(DERIVED, "split_assignments.csv"), index=False)

# Feature statistics on ALL kept rows (no labels), saved only so leakage check 4 in the training
# notebook can confirm its fitted medians/means come from training rows and NOT from all rows.
num_cols = ["_EDUCAG", "BMI", "ALC_DAYS30", "CHECKUP1", "GENHLTH", "PHYSHLTH", "MENTHLTH", "SDLONELY", "SDHFOOD1"]
med_all = clean[num_cols].median()
pd.DataFrame({"median_all_rows": med_all,
              "mean_after_median_fill_all_rows": clean[num_cols].fillna(med_all).mean()}
            ).to_csv(os.path.join(OUTPUTS, "step3_all_rows_feature_stats.csv"))

print("test file:", clean["split"].eq("test").sum(), "rows | dev file:", clean["split"].ne("test").sum(), "rows")

test file: 86793 rows | dev file: 347170 rows


In [13]:
# Step 2 check: rebuilt counts vs. the old build (INSTRUCTIONS.md section 3)
# If anything differs, stop and report.
labeled = slim[slim["DIABETE4"].isin([1, 3])]
checks = pd.DataFrame([
    ("Raw respondents",                              457670, len(df)),
    ("After keeping DIABETE4 in {1, 3}",             441934, len(labeled)),
    ("After dropping unknown age",                   433963, len(clean)),
    ("Positives",                                     64821, int(clean["target"].sum())),
    ("Positive rate (%)",                              14.9, round(clean["target"].mean() * 100, 1)),
    ("SDOH module respondents (has_sdoh)",           192706, int(clean["has_sdoh"].sum())),
    ("Columns incl. split (old build)",                   37, clean.shape[1]),
], columns=["check", "expected", "rebuilt"])
checks["match"] = checks["expected"] == checks["rebuilt"]
checks.to_csv(os.path.join(OUTPUTS, "step2_rebuild_counts.csv"), index=False)
print(checks.to_string(index=False))
print("\nALL MATCH" if checks["match"].all() else "\nMISMATCH: stop and report")

                             check  expected  rebuilt  match
                   Raw respondents  457670.0 457670.0   True
  After keeping DIABETE4 in {1, 3}  441934.0 441934.0   True
        After dropping unknown age  433963.0 433963.0   True
                         Positives   64821.0  64821.0   True
                 Positive rate (%)      14.9     14.9   True
SDOH module respondents (has_sdoh)  192706.0 192706.0   True
   Columns incl. split (old build)      37.0     37.0   True

ALL MATCH
